# Industrial Safety & Compliance Auditor

**Can a local LLM audit yard operations logs against federal safety regulations — and can you trust the rules it cites?**

This notebook audits a day of intermodal yard operations against OSHA and FRA regulations using a
locally hosted model, and then measures how often the model's legal citations are real.

That second question is the point. A compliance tool that flags the right event for an invented
reason is worse than no tool, because the citation is what a safety manager acts on. The headline
result here is not the detection score — it is the gap between how often the model is *right* and
how often it can *show its work*.

**How to re-run everything from scratch**

```bash
python code/fetch_regulations.py      # pull 29 CFR 1910 D/N/S + 49 CFR 228 from eCFR
python code/build_section_index.py    # index every real section in 29 and 49 CFR
python code/generate_yard_log.py      # regenerate the labelled synthetic log (seeded)
python code/audit_agent.py            # retrieval-grounded audit, writes json/audit_results.json
python code/audit_agent.py --no-retrieval   # ungrounded baseline, for comparison
```

The audit steps need [Ollama](https://ollama.com) running locally with `mxbai-embed-large` and
`qwen3.5:9b` pulled. Nothing leaves the machine. This notebook reads the saved outputs, so it runs
in seconds without them.

## Setup

In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 120)
ROOT = Path.cwd()

regs        = json.loads((ROOT / 'json' / 'regulations.json').read_text(encoding='utf-8'))
valid       = json.loads((ROOT / 'json' / 'valid_sections.json').read_text(encoding='utf-8'))
log         = pd.read_csv(ROOT / 'daily_yard_log.csv')
grounded    = json.loads((ROOT / 'json' / 'audit_results.json').read_text(encoding='utf-8'))
ungrounded  = json.loads((ROOT / 'json' / 'audit_results_ungrounded.json').read_text(encoding='utf-8'))

print(f"corpus     : {regs['total_chunks']} chunks, {len(regs['sections'])} sections")
print(f"section idx: {valid['total_sections']:,} real sections across {', '.join(valid['titles'])} CFR")
print(f"yard log   : {len(log)} events, {int(log.Is_Violation.sum())} labelled violations")

## Part 1 — The regulatory corpus

An earlier version of this project shipped **Title 14 CFR** as its OSHA/FRA corpus. Title 14 is
*Aeronautics and Space* — pilot certification and flight simulator qualification. 6.4 MB of aviation
law standing in for workplace safety rules.

The corpus below is the law a rail-served intermodal yard actually operates under. Each chunk keeps
the citation it came from, which is what makes a model's citation checkable later.

In [ ]:
chunks = pd.DataFrame(regs['chunks'])
summary = (chunks.groupby(['title', 'part', 'topic'])
                 .agg(sections=('section', 'nunique'), chunks=('chunk_id', 'count'),
                      chars=('char_len', 'sum'))
                 .reset_index())
summary['chars'] = (summary['chars'] / 1000).round(1).astype(str) + ' KB'
display(summary)

print("\nSections that carry the three hazards this audit looks for:")
for sec in ['228.11', '1910.22', '1910.178', '1910.333']:
    row = chunks[chunks.section == sec].head(1)
    if not row.empty:
        print(f"  {row.iloc[0]['citation']:16} {row.iloc[0]['heading']}")

## Part 2 — Ground truth

The original generator was unseeded, so the committed CSV could not be reproduced, and its random
branch emitted the same hazard patterns it was deliberately planting. Five rows matched
spill-in-a-pedestrian-zone when only one was labelled; four matched high-voltage proximity when only
one was labelled. Scoring a model against the planted rows would have counted correct catches as
false positives.

This generator is seeded, and clean rows are drawn from pools that cannot reproduce a hazard
pattern. The check below is the same one that runs inside `generate_yard_log.py` and fails the build
if it ever trips.

In [ ]:
print(log.Violation_Type.value_counts().to_string())

PED, HV = 'Main Pedestrian Crosswalk', 'High-Voltage Line B'
implied = (
    (log.Operator_Shift_Hours > 12.0)
    | ((log.Location == PED) & (log.Reported_Incident == 'Hydraulic Leak'))
    | ((log.Location == HV) & (log.Reported_Incident == 'Proximity Warning'))
)
mismatch = log[implied != log.Is_Violation.astype(bool)]
print(f"\nRows whose hazard pattern disagrees with its label: {len(mismatch)}")
print("Ground truth is clean." if mismatch.empty else "CONTAMINATED")

display(log[log.Is_Violation == 1][
    ['Log_ID', 'Equipment_Type', 'Location', 'Operator_Shift_Hours',
     'Reported_Incident', 'Violation_Type']])

## Part 3 — What the audit actually does

For each event, the pipeline embeds a description of it, retrieves the four most similar regulation
chunks by cosine similarity, and shows the model **only those excerpts**, instructing it to cite a
section from them or cite nothing.

The original build skipped this entirely. It chunked the corpus, assigned it to a variable under the
comment `#Handoff for embeddings`, and never referred to that variable again. The prompt asked the
model to consult "your regulatory knowledge base" — its own weights.

Below is what retrieval surfaced for each planted violation.

In [ ]:
g = pd.DataFrame(grounded['results'])
planted = g[g.actual == 1][['log_id', 'actual_type', 'retrieved', 'top_score']]
planted = planted.assign(retrieved=planted.retrieved.apply(lambda s: ', '.join(s)))
display(planted)

## Part 4 — Detection performance

Precision and recall over all 50 events, against the labels above. The earlier build reported
neither — it ran on four hand-picked rows, three of which were the planted violations.

Neither configuration is close to usable. Grounding roughly triples recall, but two-thirds of the
planted violations still get through, and these are blunt ones. The ungrounded run's precision of
1.00 is an artefact of near-total silence: it flagged one event in fifty.

In [ ]:
def frame(payload, label):
    m = payload['metrics']
    return pd.Series({
        'true positives': m['true_positives'], 'false positives': m['false_positives'],
        'false negatives': m['false_negatives'], 'true negatives': m['true_negatives'],
        'precision': m['precision'], 'recall': m['recall'],
        'F1': m['f1'], 'accuracy': m['accuracy'],
    }, name=label)

detect = pd.concat([frame(grounded, 'retrieval-grounded'),
                    frame(ungrounded, 'ungrounded (original)')], axis=1)
display(detect)

## Part 5 — Citation grounding: the result that matters

Detection, as Part 4 shows, is poor in both configurations. The separate question — and the one
this rebuild was aimed at — is whether the rule the auditor names is real.

Two separate checks, because conflating them would flatter retrieval:

- **exists** — is the section a real one, anywhere in 29 CFR or 49 CFR? Checked against all 16,173
  real section numbers, so citing a genuine rule that happens to fall outside this project's four
  subparts is *not* counted as fabrication.
- **grounded** — was the section among the excerpts the model was actually shown?

In [ ]:
def cites(payload, label):
    m = payload['metrics']
    return pd.Series({
        'citations given': m['citations_given'],
        'fabricated (no such section)': m['citations_fabricated'],
        'real but never retrieved': m['citations_outside_retrieved_set'],
        'citation validity': m['citation_validity'],
    }, name=label)

cite_table = pd.concat([cites(grounded, 'retrieval-grounded'),
                        cites(ungrounded, 'ungrounded (original)')], axis=1)
display(cite_table)

fig, ax = plt.subplots(figsize=(7, 4))
labels = ['retrieval-grounded', 'ungrounded (original)']
valid_pct = [(m['metrics']['citation_validity'] or 0) * 100
             for m in (grounded, ungrounded)]
bars = ax.bar(labels, valid_pct, color=['#2c7fb8', '#de2d26'], width=.55)
ax.bar_label(bars, fmt='%.0f%%', padding=3, fontsize=12, fontweight='bold')
ax.set_ylim(0, 112)
ax.set_ylabel('Citations that name a real CFR section (%)')
ax.set_title('Does the auditor cite rules that exist?', fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('citation_validity.png', dpi=200)
plt.show()

In [ ]:
#Every citation the ungrounded run invented, with what the number really is
u = pd.DataFrame(ungrounded['results'])
fabricated = u[u.citation.notna() & ~u.citation_exists.fillna(False).astype(bool)]
if fabricated.empty:
    print("The ungrounded run produced no fabricated section numbers in this run.")
else:
    display(fabricated[['log_id', 'citation_raw', 'reason']].head(15))
    print(f"\n{len(fabricated)} of {u.citation.notna().sum()} citations point at "
          f"section numbers that do not exist.")

## Limitations

Worth stating plainly, since the point of this rebuild was that the last version's claims outran
what it measured:

- **The log is synthetic.** Violations were planted by rule, so the detection task is easier than
  real yard data, where hazards are implicit and reporting is inconsistent. The detection numbers
  are a floor on difficulty, not evidence of field performance.
- **Fifty events is a small sample.** With nine violations, one missed catch moves recall by 11
  points. Treat the citation-grounding gap — a much larger effect — as the durable finding.
- **One model, one retrieval configuration.** `qwen3.5:9b` at top-k 4. No sweep over k, chunk size,
  or model, so nothing here says this configuration is optimal.
- **Retrieval quality is not measured directly.** Whether the *right* rule was retrieved is only
  observed through whether the model cited something real and on point.
- **The audit reads one event at a time.** Real fatigue and equipment-defect patterns build across a
  shift, and a per-row prompt cannot see them.